# harness

> Communication channels for multi-agent harnesses (Phase 7).

Spec: `specs/multiagent/phase7/`.

In [ ]:
#| default_exp harness

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from collections import defaultdict, deque

import numpy as np


class Blackboard:
    blocked = frozenset()                      # writers whose writes are dropped (set by a Phase 9 Monitor)

    def __init__(self, L, K=None, lam=0.0, n=None):
        self.L, self.K, self.lam, self.tRef = L, K, float(lam), 0.0
        if K is None:
            self.S, self.C = np.zeros(L), np.zeros(L)
            self.Si, self.Ci = np.zeros((n, L)), np.zeros((n, L))
        else:
            self.V, self.Tw = np.zeros((L, K)), np.zeros((L, K))
            self.Wr = np.full((L, K), -1)
            self.count = np.zeros(L, int)
        self.stats = defaultdict(lambda: dict(written=0, sent=0, received=0))

    def _age(self, t):
        if t < self.tRef:
            raise ValueError('time went backwards')
        f = np.exp(-self.lam * (t - self.tRef))
        self.S *= f
        self.C *= f
        self.Si *= f
        self.Ci *= f
        self.tRef = t

    def write(self, i, loc, value, t):
        if i in self.blocked:
            self.stats[i]['rejected'] = self.stats[i].get('rejected', 0) + 1
            return
        if self.K is None:
            self._age(t)
            self.S[loc] += value
            self.C[loc] += 1
            self.Si[i, loc] += value
            self.Ci[i, loc] += 1
        else:
            if t < self.tRef:
                raise ValueError('time went backwards')
            s = self.count[loc] % self.K
            self.V[loc, s], self.Tw[loc, s], self.Wr[loc, s] = value, t, i
            self.count[loc] += 1
            self.tRef = t
        self.stats[i]['written'] += 1

    def rebase(self, t):
        "Move the time origin to t: entries keep their ages; later calls may start again from 0."
        if self.K is None:
            self._age(t)
        else:
            if t < self.tRef:
                raise ValueError('time went backwards')
            self.Tw = self.Tw - t
        self.tRef = 0.0

    def viewBy(self, i, t, weights):
        "View with each writer's sums scaled by weights[writer]; writer i left out (running-sum mode only)."
        if self.K is not None:
            raise ValueError('viewBy needs K = None')
        self._age(t)
        w = np.asarray(weights, float).copy()
        w[i] = 0.0
        return self._clean(w @ self.Si, w @ self.Ci)

    def _clean(self, S, C):
        S, C = S.copy(), C.copy()
        m = C < 1e-12
        S[m] = 0
        C[m] = 0
        return S, C

    def view(self, i, t):
        if self.K is None:
            self._age(t)
            S, C = (self.S.copy(), self.C.copy()) if i < 0 else (self.S - self.Si[i], self.C - self.Ci[i])
            return self._clean(S, C)
        if t < self.tRef:
            raise ValueError('time went backwards')
        valid = np.arange(self.K)[None, :] < np.minimum(self.count, self.K)[:, None]
        wt = np.exp(-self.lam * (t - self.Tw)) * valid * (self.Wr != i)
        return self._clean((wt * self.V).sum(1), wt.sum(1))


class MessageBus:
    blocked = frozenset()                      # senders whose messages are dropped (set by a Phase 9 Monitor)

    def __init__(self, n, k=2, period=10.0, radius=np.inf, rngs=None):
        self.k, self.period, self.radius, self.rngs = int(k), float(period), radius, rngs
        self.window = [-1] * n
        self.inbox = [deque() for _ in range(n)]
        self.stats = defaultdict(lambda: dict(written=0, sent=0, received=0))

    def due(self, i, t):
        m = int(t // self.period)
        if m > self.window[i]:
            self.window[i] = m
            return True
        return False

    def send(self, world, i, loc, value, t):
        if i in self.blocked:
            self.stats[i]['rejected'] = self.stats[i].get('rejected', 0) + 1
            return []
        near = sorted(world.neighbours(i, self.radius))
        if not near:
            return []
        to = self.rngs[i].choice(near, min(self.k, len(near)), replace=False)
        for r in to:
            self.inbox[r].append((i, int(loc), float(value), float(t)))
            self.stats[r]['received'] += 1
        self.stats[i]['sent'] += len(to)
        return [int(r) for r in to]

    def take(self, i):
        out = list(self.inbox[i])
        self.inbox[i].clear()
        return out


class Lead:
    def __init__(self, L, n, K=None, lam=0.01, cUCB=1.0, H=10.0):
        self.table = Blackboard(L, K, lam, n)
        self.assigned = [None] * n
        self.cUCB, self.H = cUCB, H
        self.stats = self.table.stats

    def report(self, i, loc, value, nHere, t):
        self.table.write(i, loc, value * max(int(nHere), 1), t)

    def assign(self, i, loc, travel, t):
        S, C = self.table.view(-1, t)
        unknown = C == 0
        if unknown.any():
            idx = np.flatnonzero(unknown)
            j = int(idx[np.argmin(travel[idx])])
        else:
            a = np.bincount([x for k, x in enumerate(self.assigned) if k != i and x is not None], minlength=len(C))
            Ihat = S / C
            score = (Ihat / (a + 1) + self.cUCB * np.sqrt(np.log1p(C.sum()) / C)) * self.H / (self.H + travel)
            j = int(np.argmax(score))
        self.assigned[i] = j
        return j

## Tests

In [ ]:
import numpy as np
from fastcore.test import test_eq, test_close
from explRL.harness import Blackboard, MessageBus, Lead
from explRL.patchworld import TeamPatchWorld, ringLayout
from explRL.multi import spawnRngs, runTeam
from explRL.team_refs import idealFree

# AC-3 Blackboard
bb = Blackboard(3, K=2, lam=0.0, n=2)
bb.write(0, 0, 1.0, 0.0); bb.write(1, 0, 3.0, 1.0); bb.write(0, 0, 5.0, 2.0)
S, C = bb.view(0, 2.0)
test_close(S[0], 3.0); test_eq(int(C[0]), 1)
bb2 = Blackboard(3, K=None, lam=0.0, n=2)
for k in range(1000):
    bb2.write(k % 2, 0, 1.0, float(k))
S2, C2 = bb2.view(0, 999.0)
bb3 = Blackboard(3, K=1000, lam=0.0, n=2)
for k in range(1000):
    bb3.write(k % 2, 0, 1.0, float(k))
S3, C3 = bb3.view(0, 999.0)
test_close(S2, S3); test_close(C2, C3)
bbL = Blackboard(2, lam=0.05, n=1)
bbL.write(0, 0, 1.0, 0.0)
_, Cb = bbL.view(-1, np.log(2) / 0.05)
test_close(Cb[0], 0.5, eps=1e-6)
try:
    bbL.view(0, -1.0)
    raise AssertionError('expected ValueError')
except ValueError:
    pass

# AC-4 MessageBus
w = TeamPatchWorld(ringLayout(8), nPatches=2, nDanger=0, patchSites=[0,4], dangerSites=[], intake=(1.0, 1.0), Tmove=None)
w.reset(0, 4)
bus = MessageBus(4, k=2, period=10.0, radius=0.0, rngs=spawnRngs(9, 4))
w.loc = [0, 0, 4, 4]
assert bus.due(0, 0.0) and not bus.due(0, 5.0) and bus.due(0, 10.0)
rec = bus.send(w, 0, 1, 2.5, 0.0)
assert len(rec) <= 2 and all(w.loc[r] == 0 for r in rec)
msgs = bus.take(rec[0])
test_eq(len(msgs), 1); test_eq(msgs[0][1], 1)

# AC-5 Lead
lead = Lead(4, 3, lam=0.0, cUCB=0.0, H=10.0)
lead.report(0, 0, 2.0, 1, 0.0)
lead.report(0, 1, 1.0, 1, 0.0)
for p in (2, 3):
    lead.report(0, p, 0.0, 1, 0.0)
travel = np.ones(4)
test_eq(lead.assign(0, 0, travel, 1.0), 0)
test_eq(lead.assign(1, 0, travel, 1.0), 0)
test_eq(lead.assign(2, 0, travel, 1.0), 1)
n_ifd, y = idealFree(TeamPatchWorld(ringLayout(8), nPatches=2, nDanger=0, patchSites=[0,4], dangerSites=[], intake=(2.0,1.0)), 3)
test_eq(list(n_ifd[:2]), [2, 1])

# AC-3 (more): own entries left out in running-sum mode; lam = 0 never fades
bb4 = Blackboard(2, lam=0.0, n=2)
bb4.write(0, 0, 2.0, 0.0); bb4.write(1, 0, 4.0, 0.0)
S4, C4 = bb4.view(0, 1e6)
test_close(S4[0], 4.0); test_close(C4[0], 1.0)
# K = None equals a window K = 1000 for the same 1 000 writes, with and without fading, for every reader
for lam in (0.0, 0.05):
    a, b = Blackboard(3, K=None, lam=lam, n=2), Blackboard(3, K=1000, lam=lam, n=2)
    for k in range(1000):
        for x in (a, b):
            x.write(k % 2, k % 3, float(k % 7), k / 10)
    for i in (-1, 0, 1):
        Sa, Ca = a.view(i, 100.0); Sb, Cb2 = b.view(i, 100.0)
        test_close(Sa, Sb, eps=1e-6); test_close(Ca, Cb2, eps=1e-6)
# weights below 1e-12 read as 0 (exp(-0.01 x 2800) < 1e-12)
bbF = Blackboard(2, lam=0.01, n=1)
bbF.write(0, 0, 1.0, 0.0)
SF, CF = bbF.view(-1, 2800.0)
test_eq((float(SF[0]), float(CF[0])), (0.0, 0.0))
# the windowed mode also rejects earlier times
bbW = Blackboard(2, K=3, lam=0.01, n=1)
bbW.write(0, 0, 1.0, 5.0)
for f in (lambda: bbW.view(0, 4.0), lambda: bbW.write(0, 0, 1.0, 4.0)):
    try:
        f(); raise AssertionError('expected ValueError')
    except ValueError:
        pass

# AC-4 (more): radius inf reaches anyone but the sender; take returns each report once, oldest first;
# the same seed gives the same recipients
busI = MessageBus(4, k=3, radius=np.inf, rngs=spawnRngs(9, 4))
test_eq(sorted(busI.send(w, 0, 1, 1.0, 0.0)), [1, 2, 3])
for t, loc in ((1.0, 5), (2.0, 6)):
    busI.send(w, 0, loc, 1.0, t)
test_eq([m[1] for m in busI.take(1)], [1, 5, 6]); test_eq(busI.take(1), [])
test_eq(busI.stats[0]['sent'], 9); test_eq(busI.stats[2]['received'], 3)
recs = [MessageBus(4, k=1, rngs=spawnRngs(3, 4)).send(w, 2, 0, 1.0, 0.0) for _ in range(2)]
test_eq(recs[0], recs[1])

# AC-5 (more): unknown locations first, nearest first; a follower's own previous assignment is not counted
lu = Lead(4, 2, lam=0.0, cUCB=0.0)
lu.report(0, 0, 1.0, 1, 0.0)
test_eq(lu.assign(0, 0, np.array([0., 3., 1., 2.]), 1.0), 2)
lo = Lead(2, 2, lam=0.0, cUCB=0.0)
lo.report(0, 0, 2.0, 1, 0.0); lo.report(0, 1, 1.5, 1, 0.0)
test_eq(lo.assign(0, 0, np.ones(2), 1.0), 0)
test_eq(lo.assign(0, 0, np.ones(2), 2.0), 0)   # its own assignment to 0 is not counted: 2/1 > 1.5
test_eq(lo.assign(1, 0, np.ones(2), 2.0), 1)   # follower 0 holds 0: 2/2 < 1.5

# rebase keeps every entry's age (both storage modes), and times may start again from 0
for K in (None, 3):
    b1, b2 = Blackboard(2, K=K, lam=0.05, n=2), Blackboard(2, K=K, lam=0.05, n=2)
    for b in (b1, b2):
        b.write(0, 0, 1.0, 10.0); b.write(1, 0, 3.0, 20.0); b.write(1, 1, 2.0, 25.0)
    S1, C1 = b1.view(0, 50.0)
    b2.rebase(30.0)
    S2, C2 = b2.view(0, 20.0)
    test_close(S1, S2, eps=1e-9); test_close(C1, C2, eps=1e-9)
    b2.write(0, 1, 1.0, 21.0)

# blocked writers are dropped and counted; viewBy with all weights 1 equals view, with one weight 0 drops that writer
bb = Blackboard(3, lam=0.01, n=3)
bb.write(0, 0, 1.0, 0.0); bb.write(1, 0, 3.0, 1.0); bb.write(2, 1, 2.0, 2.0)
test_close(bb.viewBy(0, 5.0, [1, 1, 1]), bb.view(0, 5.0))
S0, C0 = bb.viewBy(0, 5.0, [1, 0, 1]); test_close((S0[0], C0[0]), (0.0, 0.0)); assert C0[1] > 0
bb.blocked = {1}; bb.write(1, 2, 9.0, 6.0)
test_eq(bb.stats[1]['rejected'], 1); test_eq(float(bb.view(-1, 6.0)[1][2]), 0.0)
try:
    Blackboard(3, K=2, n=2).viewBy(0, 0.0, [1, 1]); raise AssertionError('expected ValueError')
except ValueError:
    pass
wb = TeamPatchWorld(ringLayout(8), nPatches=1, nDanger=0, patchSites=[0], dangerSites=[], intake=(1.0,), Tmove=None)
wb.reset(0, 3)
mb = MessageBus(3, k=2, rngs=spawnRngs(5, 3)); mb.blocked = {0}
test_eq(mb.send(wb, 0, 1, 1.0, 0.0), []); test_eq(mb.stats[0]['rejected'], 1); test_eq(mb.take(1), [])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()